Here’s the Python version of that C++ abstract `material` class, with **why it exists**, **how it maps from C++ to Python**, and the math behind ray scattering.

---

# 1. Purpose of `material`

In ray tracing, when a ray hits an object:

* it may **bounce** (scatter),
* it may **lose energy** (attenuation),
* or it may be **absorbed**.

Mathematically:

If incoming ray is:

$$
R(t) = O + tD
$$

where:

* $O$ = origin
* $D$ = direction
* $t$ = distance parameter

After hitting a surface:

$$
R_s(t) = P + tD_s
$$

where:

* $P$ = hit point
* $D_s$ = scattered direction

The color becomes:

$$
C_{new} = A \cdot C_{old}
$$

where:

* $A$ = attenuation factor

---

# 2. C++ → Python Conversion

C++ uses:

```cpp
virtual bool scatter(...) const = 0;
```

Python uses:

```python
from abc import ABC, abstractmethod
```

because Python has **Abstract Base Classes (ABC)**.

---

# Python Version

```python
from abc import ABC, abstractmethod

class Material(ABC):
    """
    Abstract base class for all materials.
    Every material must define how rays scatter.
    """

    @abstractmethod
    def scatter(self, r_in, rec):
        """
        Parameters:
            r_in : incoming ray
            rec  : hit record

        Returns:
            (did_scatter, attenuation, scattered_ray)
        """
        pass
```

---

# 3. Step-by-step Learning Table

| Step | Code                              | Purpose                       | Formula       | Meaning              |
| ---- | --------------------------------- | ----------------------------- | ------------- | -------------------- |
| 1    | `from abc import ABC`             | Makes abstract class possible | —             | Base for inheritance |
| 2    | `class Material(ABC)`             | Creates parent class          | —             | Blueprint only       |
| 3    | `@abstractmethod`                 | Forces child implementation   | —             | Like C++ virtual     |
| 4    | `scatter(self, r_in, rec)`        | Defines scattering behavior   | $R(t)=O+tD$   | Incoming ray         |
| 5    | `return (bool, attenuation, ray)` | Returns result                | $C'=A\cdot C$ | New color            |

---

# 4. Why abstract?

Without abstraction:

```python
if material_type == "metal":
```

bad because:

* too many `if-else`
* hard to extend

With abstraction:

```python
material.scatter()
```

good because:

* cleaner
* expandable
* object-oriented

This follows:

$$
Behavior = f(MaterialType)
$$

instead of:

$$
Behavior = f(type_1,type_2,...,type_n)
$$

---

# 5. Example Child Class: Lambertian (Diffuse)

Diffuse reflection scatters randomly.

Formula:

$$
D_s = N + random_unit_vector()
$$

where:

* $N$ = surface normal

Python:

```python
import random

class Lambertian(Material):
    def __init__(self, albedo):
        self.albedo = albedo

    def scatter(self, r_in, rec):
        scatter_direction = rec.normal + random_unit_vector()

        scattered = Ray(rec.p, scatter_direction)
        attenuation = self.albedo

        return True, attenuation, scattered
```

---

Learning table:

| Part                   | Formula | Purpose          |
| ---------------------- | ------- | ---------------- |
| `rec.normal`           | $N$     | Surface normal   |
| `random_unit_vector()` | $U$     | Random direction |
| `N+U`                  | $D_s$   | Diffuse scatter  |
| `albedo`               | $A$     | Surface color    |

Full:

$$
D_s = N + U
$$

Attenuation:

$$
C_{out} = A \cdot C_{in}
$$

---

# 6. Example Child Class: Metal

Metal reflects.

Reflection formula:

$$
R = V - 2(V \cdot N)N
$$

Python:

```python
class Metal(Material):
    def __init__(self, albedo):
        self.albedo = albedo

    def scatter(self, r_in, rec):
        reflected = reflect(r_in.direction, rec.normal)

        scattered = Ray(rec.p, reflected)
        attenuation = self.albedo

        return True, attenuation, scattered
```

---

Table:

| Formula            | Meaning         |
| ------------------ | --------------- |
| $V$                | incoming vector |
| $N$                | normal          |
| $V \cdot N$        | projection      |
| $R=V-2(V\cdot N)N$ | reflection      |

---

# 7. How it works in renderer

```python
material = Lambertian(color(0.8, 0.3, 0.3))

did_scatter, attenuation, scattered = material.scatter(ray_in, hit_record)
```

Dynamic dispatch:

If object is Lambertian:

$$
scatter() \to Lambertian.scatter()
$$

If Metal:

$$
scatter() \to Metal.scatter()
$$

This is:

$$
ParentReference \rightarrow ChildBehavior
$$

(polymorphism)

---

# Final concept map

```text
Material (Abstract)
│
├── Lambertian
│      scatter() → diffuse
│
├── Metal
│      scatter() → reflection
│
└── Dielectric
       scatter() → refraction
```

Tree form:

$$
Material \rightarrow {Lambertian, Metal, Dielectric}
$$

Each implements:

$$
scatter(r_{in}, rec)
$$

This is the core idea behind physically-based ray tracing.


In [1]:
from abc import ABC, abstractmethod

class Material(ABC):
    """
    Abstract base class for all materials.
    Every material must define how rays scatter.
    """

    @abstractmethod
    def scatter(self, r_in, rec):
        """
        Parameters:
            r_in : incoming ray
            rec  : hit record

        Returns:
            (did_scatter, attenuation, scattered_ray)
        """
        pass